# RAG on Azure — Day 13: Metadata-Aware Retrieval at Scale 🗂️

**The problem:** semantic similarity has no concept of *validity*. Ask
*"How many days must I be in the office?"* against a corpus that contains
**three versions** of the Remote Work Policy and the top hits will be... all
three. v1 says 1 day, v2 says 3 days, v3 says 2 days. The embeddings are
nearly identical — semantically they ARE the same content. Which one is
*current* is not a semantic question. It's a **structured data** question.

**Today's fix:** a richer index schema. During ingestion we parse each
document's metadata (`docType`, `version`, `effectiveDate`, `author`,
`department`, `status`) into **filterable, facetable fields**, compute an
`isLatest` flag per document family, and combine vector search with **OData
filters** at query time. Vectors find *what's relevant*; filters decide
*what's eligible*.

**Pipeline today:**

1. Load 7 corporate PDFs (3 versions of one policy, 2 of another, a memo, a spec) — each carries a Document Control block on page 1.
2. Parse that block into structured metadata at ingestion; compute `isLatest` per document title.
3. Build an index where metadata fields are `filterable` / `facetable` / `sortable`.
4. Run vector search **with** filters: latest-only, by author, by department, by date range — and facet the corpus like an e-commerce sidebar.

**Done when:** you can slice the corpus by structured attributes during
retrieval — and the "office days" question returns exactly one (correct) policy.

## 1. Setup & configuration

Same pattern as Days 1–12 — `common/` helpers if you've been following along,
inline fallbacks if not.

In [ ]:
import os, json, re, time, hashlib
from collections import defaultdict

try:
    from common.azure_clients import load_config, get_openai_client, get_search_client
except ImportError:
    from openai import AzureOpenAI
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    def load_config(path="config.json"):
        if os.path.exists(path):
            with open(path) as f:
                return json.load(f)
        return {
            "openai_endpoint":      os.environ["AZURE_OPENAI_ENDPOINT"],
            "openai_key":           os.environ["AZURE_OPENAI_KEY"],
            "search_endpoint":      os.environ["AZURE_SEARCH_ENDPOINT"],
            "search_key":           os.environ["AZURE_SEARCH_KEY"],
            "chat_deployment":      os.environ.get("AZURE_CHAT_DEPLOYMENT", "gpt-4o-mini"),
            "embedding_deployment": os.environ.get("AZURE_EMBED_DEPLOYMENT", "text-embedding-3-small"),
        }

    def get_openai_client(config):
        return AzureOpenAI(azure_endpoint=config["openai_endpoint"],
                           api_key=config["openai_key"], api_version="2024-10-21")

    def get_search_client(config, index_name):
        return SearchClient(endpoint=config["search_endpoint"], index_name=index_name,
                            credential=AzureKeyCredential(config["search_key"]))

config = load_config()
openai = get_openai_client(config)

INDEX_NAME = "rag-corp-day13"
DATA_DIR   = "data-day13"

print("Config loaded. Chat:", config["chat_deployment"],
      "| Embeddings:", config["embedding_deployment"])

## 2. Ingestion — extract metadata from the documents themselves

Every PDF in this corpus carries a **Document Control block** on page 1
(Document Type / Version / Effective Date / Author / Department / Status) —
standard practice in regulated companies, and exactly the kind of structure
real ingestion pipelines mine. We parse it instead of inventing metadata.

In [ ]:
from pypdf import PdfReader

META_LABELS = ["Document Type", "Version", "Effective Date",
               "Author", "Department", "Status"]

def parse_document_control(first_page_text):
    """The control table extracts as alternating label/value lines."""
    lines = [l.strip() for l in first_page_text.splitlines() if l.strip()]
    meta = {}
    for i, line in enumerate(lines):
        if line in META_LABELS and i + 1 < len(lines):
            meta[line] = lines[i + 1]
    return meta

def load_pdf(path):
    reader = PdfReader(path)
    pages = [p.extract_text() for p in reader.pages]
    title = pages[0].splitlines()[0].strip()
    return title, parse_document_control(pages[0]), "\n".join(pages)

docs = []
for fname in sorted(os.listdir(DATA_DIR)):
    if not fname.endswith(".pdf"):
        continue
    title, meta, full_text = load_pdf(os.path.join(DATA_DIR, fname))
    docs.append({"file": fname, "title": title, "meta": meta, "text": full_text})
    print(f"{fname:34s} | {meta.get('Document Type','?'):14s} | "
          f"v{meta.get('Version','?'):4s} | {meta.get('Effective Date','?')} | "
          f"{meta.get('Author','?'):14s} | {meta.get('Department','?')}")

### The `isLatest` rule

The single most useful piece of metadata is one **we compute**, not parse:
group documents by title, sort by version, flag only the highest. Superseded
versions stay in the index (audit, "what changed?" questions) but default
retrieval will exclude them with a one-line filter.

In [ ]:
families = defaultdict(list)
for d in docs:
    families[d["title"]].append(d)

for title, members in families.items():
    members.sort(key=lambda d: float(d["meta"]["Version"]))
    for m in members:
        m["is_latest"] = (m is members[-1])
    print(f"{title}: {len(members)} version(s) -> latest is v{members[-1]['meta']['Version']}")

## 3. Chunk — and stamp every chunk with its document's metadata

Chunking is paragraph-per-section as before. The key habit: **metadata is
copied onto every chunk**, because chunks — not documents — are what the index
stores and what filters act on.

In [ ]:
def split_sections(text, title):
    """Sections = a heading line followed by body lines."""
    body = text.split("Status", 1)[-1]            # skip the control block
    body = body.split("\n", 1)[-1]                # drop the status value line
    lines = body.splitlines()
    sections, current_head, current = [], None, []
    for line in lines:
        # headings in this corpus are short title-case lines without a period
        if line and len(line) < 60 and not line.endswith(".") and line[0].isupper() \
           and not line[0].isdigit() and len(line.split()) <= 6:
            if current_head and current:
                sections.append((current_head, " ".join(current).strip()))
            current_head, current = line.strip(), []
        else:
            current.append(line.strip())
    if current_head and current:
        sections.append((current_head, " ".join(current).strip()))
    return [(h, b) for h, b in sections if len(b) > 40]

records = []
for d in docs:
    for heading, body_text in split_sections(d["text"], d["title"]):
        records.append({
            "id":            hashlib.md5(f"{d['file']}-{heading}".encode()).hexdigest()[:12],
            "content":       f"{d['title']} — {heading}\n{body_text}",
            "title":         d["title"],
            "section":       heading,
            "source":        d["file"],
            "docType":       d["meta"]["Document Type"],
            "department":    d["meta"]["Department"],
            "author":        d["meta"]["Author"],
            "version":       float(d["meta"]["Version"]),
            "effectiveDate": d["meta"]["Effective Date"] + "T00:00:00Z",
            "status":        d["meta"]["Status"],
            "isLatest":      d["is_latest"],
        })

print(f"{len(records)} chunks from {len(docs)} documents")
print("\nSample record metadata:")
print({k: v for k, v in records[0].items() if k not in ("content",)})

## 4. The Day 13 index schema

This is the heart of today. Compare with the minimal Day 2 schema
(id + content + vector): every metadata field is now declared with the
capabilities queries will need —

| Field | filterable | facetable | sortable | why |
|---|---|---|---|---|
| `docType`, `department`, `author`, `status` | ✅ | ✅ | | slice + count the corpus |
| `version` | ✅ | | ✅ | "latest version" logic, ordering |
| `effectiveDate` | ✅ | ✅ | ✅ | date-range queries |
| `isLatest` | ✅ | | | the default-retrieval guard |

**Schema design is a commitment:** in Azure AI Search you cannot make an
existing field filterable later without rebuilding the index. Decide your
slicing dimensions *before* you have a million documents.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchField, SearchFieldDataType,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
)
from azure.core.credentials import AzureKeyCredential

EMBED_DIMS = 1536

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SearchableField(name="title", type=SearchFieldDataType.String, filterable=True),
    SimpleField(name="section",       type=SearchFieldDataType.String,  filterable=True),
    SimpleField(name="source",        type=SearchFieldDataType.String,  filterable=True),
    SimpleField(name="docType",       type=SearchFieldDataType.String,  filterable=True, facetable=True),
    SimpleField(name="department",    type=SearchFieldDataType.String,  filterable=True, facetable=True),
    SimpleField(name="author",        type=SearchFieldDataType.String,  filterable=True, facetable=True),
    SimpleField(name="version",       type=SearchFieldDataType.Double,  filterable=True, sortable=True),
    SimpleField(name="effectiveDate", type=SearchFieldDataType.DateTimeOffset,
                filterable=True, facetable=True, sortable=True),
    SimpleField(name="status",        type=SearchFieldDataType.String,  filterable=True, facetable=True),
    SimpleField(name="isLatest",      type=SearchFieldDataType.Boolean, filterable=True),
    SearchField(name="contentVector",
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True, vector_search_dimensions=EMBED_DIMS,
                vector_search_profile_name="vp"),
]
vs = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw")],
    profiles=[VectorSearchProfile(name="vp", algorithm_configuration_name="hnsw")],
)

ic = SearchIndexClient(config["search_endpoint"], AzureKeyCredential(config["search_key"]))
try:
    ic.delete_index(INDEX_NAME)
except Exception:
    pass
ic.create_index(SearchIndex(name=INDEX_NAME, fields=fields, vector_search=vs))
print(f"Index '{INDEX_NAME}' created with {len(fields)} fields.")

In [ ]:
def embed(texts, batch=16):
    out = []
    for i in range(0, len(texts), batch):
        resp = openai.embeddings.create(model=config["embedding_deployment"],
                                        input=texts[i:i+batch])
        out.extend(d.embedding for d in resp.data)
    return out

vectors = embed([r["content"] for r in records])
search  = get_search_client(config, INDEX_NAME)
search.upload_documents(documents=[{**r, "contentVector": v}
                                   for r, v in zip(records, vectors)])
time.sleep(2)
print(f"Uploaded {len(records)} chunks.")

## 5. Query helpers — vector search **+** OData filters

One new argument does all the work: `filter=`. Azure AI Search applies the
filter **before** vector scoring, so excluded documents never compete for the
top-k — this is *pre-filtering*, which is what you want for correctness.

In [ ]:
from azure.search.documents.models import VectorizedQuery

SELECT = ["title", "section", "source", "version", "author",
          "department", "effectiveDate", "isLatest", "content"]

def retrieve(question, flt=None, k=3):
    qv = embed([question])[0]
    results = search.search(
        search_text=None,
        vector_queries=[VectorizedQuery(vector=qv, k_nearest_neighbors=k,
                                        fields="contentVector")],
        filter=flt, select=SELECT, top=k,
    )
    return list(results)

def answer(question, hits):
    context = "\n\n---\n\n".join(
        f"[{h['title']} v{h['version']} | {h['section']} | effective {h['effectiveDate'][:10]}]\n{h['content']}"
        for h in hits)
    resp = openai.chat.completions.create(
        model=config["chat_deployment"], temperature=0,
        messages=[
            {"role": "system",
             "content": "Answer ONLY from the provided context. Cite the document "
                        "title and version you used. If sources conflict, say so explicitly."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ])
    return resp.choices[0].message.content.strip()

def show(question, flt=None, k=3, label=""):
    hits = retrieve(question, flt, k)
    print(f"-- {label or ('filter: ' + (flt or 'NONE'))} --")
    for h in hits:
        print(f"   {h['title']} v{h['version']} / {h['section']} "
              f"(latest={h['isLatest']}, {h['author']})")
    print(f"   Answer: {answer(question, hits)}\n")

## 6. Demo A — the version trap (today's headline)

Three Remote Work Policies in the index, three different answers
(1 day → 3 days → 2 days). Watch the unfiltered query drown in its own history,
then watch one filter fix it.

In [ ]:
Q = "How many days per week am I required to be in the office?"
print("=" * 88); print(f"Q: {Q}\n")
show(Q, flt=None, label="UNFILTERED — all versions compete")
show(Q, flt="isLatest eq true", label="FILTERED — isLatest eq true")

## 7. Demo B — "from author X"

Pure semantic search can't reliably answer *"what has Daniel Osei published?"* —
author names barely register in embeddings. A filter makes it exact.

In [ ]:
Q = "What rules has this person put in place that affect all employees?"
print("=" * 88); print(f"Q: {Q}\n")
show(Q, flt="author eq 'Daniel Osei' and isLatest eq true",
     label="author eq 'Daniel Osei' and isLatest eq true", k=4)

## 8. Demo C — compound slices: department + date range

Realistic compliance-style query: *current Finance rules effective in 2026*.
Filters compose with `and` / `or` / comparison operators — this is OData, not
a special DSL.

In [ ]:
Q = "What are the current meal reimbursement limits?"
print("=" * 88); print(f"Q: {Q}\n")
show(Q, flt="department eq 'Finance' and effectiveDate ge 2026-01-01T00:00:00Z",
     label="department eq 'Finance' and effectiveDate ge 2026-01-01")

## 9. Demo D — superseded versions are a feature, not waste

Because old versions stay indexed, "what changed?" questions become answerable —
just point the filter AT the history instead of away from it.

In [ ]:
Q = "How did the office attendance requirement change across policy versions?"
print("=" * 88); print(f"Q: {Q}\n")
show(Q, flt="title eq 'Remote Work Policy'", k=6,
     label="title eq 'Remote Work Policy' (ALL versions, on purpose)")

## 10. Facets — let users see the shape of the corpus

Facets return **value counts** for facetable fields alongside results — the
"sidebar" pattern from e-commerce. For a RAG app this powers UI like
*"filter by department"* without any extra queries.

In [ ]:
results = search.search(search_text="*",
                        facets=["docType", "department", "status"], top=0)
for facet_name, buckets in results.get_facets().items():
    print(f"{facet_name}:")
    for b in buckets:
        print(f"   {b['value']}: {b['count']} chunks")

## 11. Takeaways

- **"Current" is not a semantic concept.** Embeddings see three policy versions
  as near-duplicates; only structured metadata knows which one is in force.
  The most dangerous RAG answer is a confident answer from a superseded document.
- **`isLatest` is computed, not found.** The highest-value metadata often comes
  from ingestion-time logic (version grouping) rather than the document itself.
- **Schema is destiny.** Filterable/facetable/sortable must be declared up
  front in Azure AI Search — design slicing dimensions before scale arrives.
- **Filters and vectors divide the labor.** Pre-filtering decides *eligibility*,
  vector scoring decides *relevance*. Together they make a 7-document demo work
  exactly like a 700,000-document corpus would.
- **Stacking so far:** contextual chunks (Day 12) put document identity *inside*
  the vector; metadata (Day 13) puts it *beside* the vector where it can be
  filtered exactly. Production systems want both.

### Next up — Day 14: the RAG evaluation harness
We've now built six retrieval upgrades (Days 8–13) on top of basic RAG — and
"it looks better in demos" is not evidence. Day 14 builds the eval set and
metrics to **prove** which techniques actually help, and by how much.